In [4]:
import pandas as pd
from pathlib import Path

In [ ]:
%pip install leia-br
from LeIA import SentimentIntensityAnalyzer

In [6]:
# Diretório base e utilitário para dados
BASE_DIR = Path.cwd().resolve()
if not (BASE_DIR / 'Banco_de_Dados_PII3_AWS').exists() and (BASE_DIR.parent / 'Banco_de_Dados_PII3_AWS').exists():
    BASE_DIR = BASE_DIR.parent
DATA_DIR = BASE_DIR / 'Banco_de_Dados_PII3_AWS'
if not DATA_DIR.exists():
    raise FileNotFoundError(f"Diretório de dados não encontrado: {DATA_DIR}")

def get_data_path(nome_arquivo):
    return DATA_DIR / nome_arquivo

In [7]:
# Inicializa o analisador
analyzer = SentimentIntensityAnalyzer()

def classificar_sentimento_numerico(score):
    if score <= -0.6:
        return 1 # Muito Negativo
    elif -0.6 < score <= -0.2:
        return 2 # Negativo
    elif -0.2 < score <= 0.2:
        return 3 # Neutro
    elif 0.2 < score <= 0.6:
        return 4 # Positivo
    else:
        return 5 # Muito Positivo

def aplicar_leia_sentimento(df, nome_coluna_texto):
    if (nome_coluna_texto in df.columns):
        print("Analisando sentimentos...")
        
        # O LeIA retorna um dicionário, pegamos apenas o 'compound'
        df['sentimento_score'] = df[nome_coluna_texto].apply(
            lambda x: analyzer.polarity_scores(str(x))['compound'] if pd.notnull(x) else 0.0
        )
        
        # Aplica a classificação inteira
        df['review_score_leia'] = df['sentimento_score'].apply(classificar_sentimento_numerico)
    else:
        print(f"Erro: Coluna '{nome_coluna_texto}' não encontrada.")
    return df

In [ ]:
df = pd.read_csv(get_data_path('avaliacoes.csv'))
df = aplicar_leia_sentimento(df, 'review_comment_message')
df.to_csv(get_data_path('avaliacoes.csv'), index=False)